# PSMNIST with LMU

In [ ]:
from __future__ import annotations
from typing import Dict, Any

import torch
import os
from pathlib import Path

from psmnist_task import PSMNISTTask
from src.train_utils.trainer import Trainer

### Configuration
We define a set of hyperparameters and configurations for the training process.
This includes data paths, batch sizes, learning rates, and model-specific parameters.
Using unified block factory from src.utils.block_factory for consistent configuration.

In [ ]:
from src.utils.block_factory import make_lmu_block_cfg_ctor

In [ ]:
current_dir = Path.cwd()
project_root = current_dir.parent.parent.parent
data_root = str(project_root / "src" / "datasets" / "psmnist" / "data")


def get_args() -> Dict[str, Any]:
    args: Dict[str, Any] = {
        "data_root": data_root,
        "batch": 128,
        "data_loader_kwargs": {
            "num_workers": 0,
            "use_permutation": True,
            "permutation_seed": 42,
            "normalize": "standard",
            "pin_memory": False,
            "persistent_workers": False,
        },

        "epochs": 50,
        "lr": 1e-3,
        "wd": 1e-4,
        "amp": False,
        "save_dir": "./runs/psmnist_lmu_task",
        "warmup_epochs": 5,
        "patience": 5,
        "min_delta": 0.001,
        "early_key": "accuracy",

        "d_model": 128,
        "depth": 2,
        "dropout": 0.1,
        "mlp_ratio": 2.0,
        "droppath_final": 0.0,
        "layerscale_init": 0.0,
        "residual_gain": 1.0,
        "pool": "mean",
    }

    args["block_cfg_ctor"] = make_lmu_block_cfg_ctor(
        dropout=args["dropout"],
        mlp_ratio=args["mlp_ratio"],
        droppath_final=args["droppath_final"],
        layerscale_init=args["layerscale_init"],
        residual_gain=args["residual_gain"],
        pool=args["pool"],
    )


    if torch.backends.mps.is_available():
        args["device"] = torch.device("mps")
        print("Using MPS")
    elif torch.cuda.is_available():
        args["device"] = torch.device("cuda")
    else:
        args["device"] = torch.device("cpu")
        args["amp"] = False

    return args

args = get_args()

### Training
With the configuration set up, we can now instantiate the `PSMNISTTask` and the `Trainer`.
The `fit` method on the trainer will start the training process, which includes training,
validation, and saving the best model based on the validation accuracy.

In [ ]:
from src.utils.visualization import plot_classification_history as plot_history

In [ ]:
task = PSMNISTTask()

if torch.backends.mps.is_available():
    torch.mps.set_per_process_memory_fraction(0.9)
    os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

trainer = Trainer(args=args, task=task)

best_metric, ckpt_path = trainer.fit()
print(f"Done. Best {trainer.early_key}={best_metric:.4f} @ {ckpt_path}")

In [ ]:
from src.utils.checkpoint import load_trainer_from_checkpoint

trainer = load_trainer_from_checkpoint(
    checkpoint_path=args["save_dir"] + "/best.pt",
    args=args,
    task=PSMNISTTask(),
)

history = trainer.history

plot_history(history, model_name="LMU")

In [ ]:
from src.utils.common import print_model_details

print_model_details(model=trainer.model)

### Evaluation
After training, we can evaluate the best model on the test set.
We load the best model from the checkpoint and then run the evaluation.
The results, including accuracy, are printed.

In [ ]:
from src.eval.eval_utils import evaluate_classification_model as evaluate_best_model

logits_test, labels_test = evaluate_best_model(
    args=args,
    task=PSMNISTTask(),
    best_model_path=f"{args['save_dir']}/best.pt",
    num_classes=10,
    use_test_set=True,
)

### Changes in dataset length

In [ ]:
from src.eval.eval_utils import evaluate_classification_model as evaluate_best_model

if torch.backends.mps.is_available():
    torch.mps.set_per_process_memory_fraction(0.9)

for frac in [0.1, 0.25, 0.5]:
    print(f"\nTraining with fraction: {frac}")

    args["fraction"] = frac
    args["save_dir"] = f"./runs/psmnist_lmu_task_frac_{int(frac*100)}"
    trainer = Trainer(args=args, task=PSMNISTTask())
    best_metric, best_path = trainer.fit()

    print(f"\nTraining complete for fraction {frac}! Best validation {trainer.early_key}: {best_metric:.4f}")
    print(f"Best model saved to: {best_path}")

    history = trainer.history

    plot_history(history, model_name="LMU")

    logits_test, labels_test = evaluate_best_model(
        args=args,
        task=PSMNISTTask(),
        best_model_path=best_path,
        num_classes=10,
        use_test_set=True,
    )

# SMNIST with LMU

In [ ]:
smnist_args = get_args()

# Change to SMNIST (Sequential MNIST without permutation)
smnist_args["data_loader_kwargs"]["use_permutation"] = False
smnist_args["save_dir"] = "../smnist/runs/smnist_lmu_task"

print("=" * 70)
print("Training on SMNIST (Sequential MNIST - no permutation)")
print("=" * 70)

### Training SMNIST

In [ ]:
smnist_task = PSMNISTTask()

if torch.backends.mps.is_available():
    torch.mps.set_per_process_memory_fraction(0.9)

smnist_trainer = Trainer(args=smnist_args, task=smnist_task)

best_metric_smnist, ckpt_path_smnist = smnist_trainer.fit()
print(f"Done. Best {smnist_trainer.early_key}={best_metric_smnist:.4f} @ {ckpt_path_smnist}")

In [ ]:
from src.utils.checkpoint import load_trainer_from_checkpoint

smnist_trainer = load_trainer_from_checkpoint(
    checkpoint_path=smnist_args["save_dir"] + "/best.pt",
    args=smnist_args,
    task=PSMNISTTask(),
)

smnist_history = smnist_trainer.history

plot_history(smnist_history, model_name="LMU (SMNIST)")

In [ ]:
print("\n" + "=" * 70)
print("Evaluating SMNIST on Test Set")
print("=" * 70)

logits_test_smnist, labels_test_smnist = evaluate_best_model(
    args=smnist_args,
    task=PSMNISTTask(),
    best_model_path=f"{smnist_args['save_dir']}/best.pt",
    num_classes=10,
    use_test_set=True,
)

### Changes in dataset length (SMNIST)
Test how the model performs with different fractions of the SMNIST training data.

In [ ]:
for frac in [0.1, 0.25, 0.5]:
    print(f"\nTraining SMNIST with fraction: {frac}")

    smnist_frac_args = get_args()
    smnist_frac_args["fraction"] = frac
    smnist_frac_args["save_dir"] = f"../smnist/runs/smnist_lmu_task_frac_{int(frac*100)}"

    smnist_frac_trainer = Trainer(args=smnist_frac_args, task=PSMNISTTask())
    best_metric_frac, best_path_frac = smnist_frac_trainer.fit()

    print(f"\nTraining complete for SMNIST fraction {frac}! Best validation {smnist_frac_trainer.early_key}: {best_metric_frac:.4f}")
    print(f"Best model saved to: {best_path_frac}")

    smnist_frac_history = smnist_frac_trainer.history

    plot_history(smnist_frac_history, model_name=f"LMU (SMNIST) - {int(frac*100)}% data")

    logits_test_frac, labels_test_frac = evaluate_best_model(
        args=smnist_frac_args,
        task=PSMNISTTask(),
        best_model_path=best_path_frac,
        num_classes=10,
        use_test_set=True,
    )